# 02 — Feature Engineering

Motivate and validate eight domain features used in the production pipeline. Each feature is engineered **before** any train/test split.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%matplotlib inline

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.data import load_raw_data, clean_total_charges, add_churn_binary
from src.features import engineer_features

sns.set_theme(style="whitegrid")

In [ ]:
df_raw, _ = load_raw_data()
df = clean_total_charges(df_raw)
df = add_churn_binary(df)
df_eng = engineer_features(df)

new_cols = [
    "avg_monthly_spend", "tenure_group", "service_count", "mtm_early",
    "electronic_check", "high_value", "missing_online_sec", "tenure_x_monthly",
]
print("New columns:", new_cols)
print("Shape before:", df.shape, "| after:", df_eng.shape)

## Feature motivations & churn lift

| Feature | Rationale |
|---------|-----------|
| `avg_monthly_spend` | Normalizes lifetime spend by tenure — separates heavy early spenders from long-tenure low spend |
| `tenure_group` | Captures non-linear tenure effects in coarse buckets |
| `service_count` | Measures product stickiness via add-on services |
| `mtm_early` | Flags month-to-month customers in fragile first year |
| `electronic_check` | Payment friction proxy |
| `high_value` | Loyal, high-ARPU customers (retention priority) |
| `missing_online_sec` | Security gap for internet customers |
| `tenure_x_monthly` | Interaction: price pressure grows with tenure |

In [ ]:
def churn_lift(col, df_in):
    """Compare churn rate when binary/col feature is active vs baseline."""
    baseline = df_in["Churn_binary"].mean()
    if df_in[col].dtype == "category" or df_in[col].dtype == object:
        rates = df_in.groupby(col, observed=False)["Churn_binary"].mean()
        lift = rates - baseline
        return pd.DataFrame({"churn_rate": rates, "lift_vs_avg": lift}).sort_values("churn_rate", ascending=False)
    rates = df_in.groupby(col)["Churn_binary"].mean()
    lift = rates - baseline
    return pd.DataFrame({"churn_rate": rates, "lift_vs_avg": lift}).sort_values("churn_rate", ascending=False)

baseline = df_eng["Churn_binary"].mean()
print(f"Baseline churn: {baseline:.1%}\n")

for col in new_cols:
    print(f"--- {col} ---")
    display(churn_lift(col, df_eng).head())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for ax, col in zip(axes, new_cols):
  if col in ("mtm_early", "electronic_check", "high_value", "missing_online_sec"):
      rates = df_eng.groupby(col)["Churn_binary"].mean()
      sns.barplot(x=rates.index.astype(str), y=rates.values, ax=ax)
  elif col == "tenure_group":
      rates = df_eng.groupby("tenure_group", observed=False)["Churn_binary"].mean()
      sns.barplot(x=rates.index.astype(str), y=rates.values, ax=ax)
      ax.tick_params(axis="x", rotation=45)
  else:
      sns.boxplot(x="Churn_binary", y=col, data=df_eng, ax=ax)
  ax.set_title(col)

plt.tight_layout()
plt.show()

## Key takeaways

- `mtm_early` and `electronic_check` flag segments with **>15 pp** churn lift over baseline.
- `tenure_group` 0–6 months shows the steepest churn — validates onboarding focus from notebook 01.
- `missing_online_sec` isolates internet customers without security — a concrete upsell target.
- Engineered features are applied identically in `src/train.py` and the Streamlit demo.